# 04 FM — Embedding Model Comparison for Mineral Production Clustering

This notebook is the **foundation-model companion** to `04_mineral_deposit_clustering.ipynb`.
Where that notebook commits to a single best-in-class encoder (BGE-Large), here we treat
the choice of embedding model as a **hyperparameter** and benchmark five open-source options
on exactly the same clustering task.

**Goal:** understand how much the choice of embedding model matters — in terms of cluster
quality metrics, inter-model agreement on country similarity, and practical factors such as
model size and inference speed.

All models are downloaded from HuggingFace Hub on first run and cached locally. No API keys
are required.

## Model Zoo

| Short name | HuggingFace model ID | Dimensions | Approx. size | Notes |
|---|---|---|---|---|
| `bge-large` | `BAAI/bge-large-en-v1.5` | 1024 | ~1.3 GB | MTEB top-tier retrieval; the baseline from notebook 04 |
| `nomic-embed` | `nomic-ai/nomic-embed-text-v1.5` | 768 | ~550 MB | Open data + open weights; Matryoshka dimensions |
| `gte-large` | `Alibaba-NLP/gte-large-en-v1.5` | 1024 | ~1.3 GB | Competitive with BGE on MTEB; strong zero-shot |
| `bge-m3` | `BAAI/bge-m3` | 1024 | ~2.3 GB | Multilingual (100+ langs); handles non-English country names |
| `MiniLM` | `sentence-transformers/all-MiniLM-L6-v2` | 384 | ~80 MB | Lightweight CPU-friendly baseline |

## Pipeline

1. **Data preparation** — identical to notebook 04: load BGS production CSV, filter to
   Production records, aggregate over the most recent 5 years, build a natural-language
   profile per country.
2. **Embedding** — encode all profiles with each of the five models in turn.
3. **Clustering** — for each model's embeddings, run UMAP → HDBSCAN with fixed
   hyperparameters so results are directly comparable.
4. **Quality metrics** — silhouette score, Calinski-Harabász index, Davies-Bouldin index.
5. **Side-by-side UMAP visualisation** — 2×3 subplot grid, one panel per model.
6. **Inter-model agreement** — Spearman rank correlation of pairwise distance matrices;
   heatmap showing which models "see" the world the same way.
7. **Similarity search comparison** — query country "China", top-10 neighbours per model,
   highlighting consensus vs. disagreement.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pathlib
import time

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import normalize
from scipy.stats import spearmanr
import umap
import hdbscan

DATA_DIR = pathlib.Path("../data/bgs_data")
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"

print(f"Data path : {CSV_PATH}")
print(f"Exists    : {CSV_PATH.exists()}")

## 2. Data Preparation

Steps are identical to `04_mineral_deposit_clustering.ipynb` so results are directly
comparable — the only variable changed across sections is the embedding model.

We filter to **Production** records, restrict to the most recent five years, aggregate
country × commodity quantities (mean over the window), drop countries with fewer than two
distinct commodities, then build a short natural-language **profile sentence** for each
surviving country that the embedding model encodes into a dense vector.

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Raw rows : {len(df_raw):,}")
print(f"Columns  : {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# ── Filter & clean ────────────────────────────────────────────────────────────
df = df_raw.copy()

# Keep production records only
df = df[df["statistic_type"].str.strip().str.lower() == "production"].copy()

# Coerce numeric columns
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

# Drop rows with missing key fields
df = df.dropna(subset=["year", "quantity", "country", "commodity"])
df = df[df["quantity"] > 0]

# Restrict to the most recent 5 years available in the dataset
max_year    = int(df["year"].max())
year_cutoff = max_year - 4   # inclusive window: [max_year-4 … max_year]
df = df[df["year"] >= year_cutoff].copy()

print(f"Production rows : {len(df):,}")
print(f"Year window     : {year_cutoff} – {max_year}")
print(f"Countries       : {df['country'].nunique()}")
print(f"Commodities     : {df['commodity'].nunique()}")

In [ ]:
# ── Aggregate: country × commodity (mean quantity over the window) ─────────────
agg = (
    df.groupby(["country", "country_iso3", "commodity"], as_index=False)["quantity"]
    .mean()
    .rename(columns={"quantity": "mean_qty"})
)

# Drop countries with fewer than 2 distinct commodities
n_commodities   = agg.groupby("country")["commodity"].nunique()
valid_countries = n_commodities[n_commodities >= 2].index
agg = agg[agg["country"].isin(valid_countries)].copy()

print(f"Countries with ≥2 commodities: {agg['country'].nunique()}")

# ── Build natural-language profiles ──────────────────────────────────────────
TOP_N = 5  # number of top minerals to mention

def build_profile(group: pd.DataFrame) -> pd.Series:
    country  = group["country"].iloc[0]
    iso3     = group["country_iso3"].iloc[0] if "country_iso3" in group.columns else ""
    ranked   = group.sort_values("mean_qty", ascending=False)
    n_min    = len(ranked)
    total    = ranked["mean_qty"].sum()
    top_min  = ranked.iloc[0]["commodity"]
    top_str  = "; ".join(
        f"{row['commodity']}: {row['mean_qty']:,.0f} tonnes"
        for _, row in ranked.head(TOP_N).iterrows()
    )
    profile = (
        f"{country} produces {n_min} critical mineral{'s' if n_min != 1 else ''}. "
        f"Top production: {top_str}."
    )
    return pd.Series({
        "country"         : country,
        "iso3"            : iso3,
        "num_minerals"    : n_min,
        "total_production": total,
        "top_mineral"     : top_min,
        "profile_text"    : profile,
    })

profiles_df = (
    agg.groupby("country", group_keys=False)
    .apply(build_profile)
    .reset_index(drop=True)
)

print(f"Profile rows: {len(profiles_df)}")
profiles_df.head()

In [ ]:
# Preview a handful of profile sentences
for _, row in profiles_df.sample(5, random_state=42).iterrows():
    print(row["profile_text"])
    print()

## 3. Embedding Model Zoo

We define a **model registry** — a dict mapping short names to HuggingFace model IDs — and a
single `encode_with_model` helper that loads each model, encodes the country profiles, and
records timing and dimension metadata.

All embeddings are **L2-normalised** before clustering and similarity computations so that
cosine similarity is equivalent to the dot product and Euclidean distance behaves sensibly in
high dimensions.

> **Memory note:** BGE-M3 (~2.3 GB) and the two large models (~1.3 GB each) are large.
> If you are on a machine with less than 16 GB RAM, consider commenting out `bge-m3` or
> `gte-large` in `MODELS` below.

In [ ]:
# ── Model registry ────────────────────────────────────────────────────────────
MODELS = {
    "bge-large"  : "BAAI/bge-large-en-v1.5",
    "nomic-embed": "nomic-ai/nomic-embed-text-v1.5",
    "gte-large"  : "Alibaba-NLP/gte-large-en-v1.5",
    "bge-m3"     : "BAAI/bge-m3",
    "MiniLM"     : "sentence-transformers/all-MiniLM-L6-v2",
}


def encode_with_model(model_name: str, model_id: str, texts: list[str]):
    """Load a SentenceTransformer model, encode texts, return (embeddings, metadata).

    Parameters
    ----------
    model_name : short label used for display
    model_id   : HuggingFace Hub model identifier
    texts      : list of plain-text strings to encode

    Returns
    -------
    embs : np.ndarray, shape (n_texts, dim), L2-normalised
    meta : dict with keys dim, load_time, encode_time
    """
    print(f"\n{'='*60}")
    print(f"Loading {model_name} ({model_id}) ...")

    t0   = time.time()
    model = SentenceTransformer(model_id, trust_remote_code=True)
    load_time = time.time() - t0

    dim = model.get_sentence_embedding_dimension()
    print(f"  Dimensions : {dim}")
    print(f"  Load time  : {load_time:.1f}s")

    t0 = time.time()
    embs = model.encode(texts, show_progress_bar=True, batch_size=32,
                        convert_to_numpy=True)
    encode_time = time.time() - t0

    embs = normalize(embs, norm="l2")

    print(f"  Encode time: {encode_time:.1f}s")
    print(f"  Shape      : {embs.shape}")

    # Unload model to free memory before loading next one
    del model

    return embs, {"dim": dim, "load_time": load_time, "encode_time": encode_time}


print(f"Models registered: {list(MODELS.keys())}")

In [ ]:
# ── Encode with every model ───────────────────────────────────────────────────
# Plain profile text is used for all models so comparisons are apples-to-apples.
# (BGE's instruction-prefix trick is intentionally omitted here to keep the
# experiment controlled; add a prefix dict if you want to test that dimension.)
texts = profiles_df["profile_text"].tolist()

all_embeddings: dict[str, np.ndarray] = {}
all_metadata:   dict[str, dict]       = {}

for name, model_id in MODELS.items():
    try:
        embs, meta = encode_with_model(name, model_id, texts)
        all_embeddings[name] = embs
        all_metadata[name]   = meta
    except Exception as exc:
        print(f"  FAILED ({name}): {exc}")

print(f"\n\nSuccessfully encoded with: {list(all_embeddings.keys())}")

## 4. Clustering Comparison

For each model we run the **same two-step pipeline**:

1. **UMAP** (`n_neighbors=15`, `min_dist=0.1`, `metric=cosine`) — projects high-dimensional
   embeddings to 2-D for visualisation.  Clustering is performed in the *full* embedding
   space, not on the 2-D projection, to preserve information.
2. **HDBSCAN** (`min_cluster_size=3`, `min_samples=2`, `metric=euclidean` on L2-normalised
   vectors — equivalent to cosine distance) — discovers clusters without requiring a
   predetermined cluster count.

Quality is measured by three complementary indices:

| Metric | Better when | Measures |
|---|---|---|
| **Silhouette score** | closer to 1.0 | intra-cluster cohesion vs. inter-cluster separation |
| **Calinski-Harabász** | higher | ratio of between-cluster to within-cluster scatter |
| **Davies-Bouldin** | closer to 0 | average similarity of each cluster to its most similar neighbour |

In [ ]:
# ── UMAP + HDBSCAN for each model ────────────────────────────────────────────
UMAP_PARAMS = dict(n_components=2, n_neighbors=15, min_dist=0.1,
                   metric="cosine", random_state=42)
HDBSCAN_PARAMS = dict(min_cluster_size=3, min_samples=2,
                      metric="euclidean", cluster_selection_method="eom")

all_results:  dict[str, pd.DataFrame] = {}   # per-model result DataFrames
all_umap_2d:  dict[str, np.ndarray]   = {}   # 2-D projections
all_labels:   dict[str, np.ndarray]   = {}   # cluster label arrays
cluster_stats: list[dict]             = []   # summary row per model

for name, embs in all_embeddings.items():
    meta = all_metadata[name]
    print(f"\n--- {name} ---")

    # UMAP
    reducer   = umap.UMAP(**UMAP_PARAMS)
    umap_2d   = reducer.fit_transform(embs)
    all_umap_2d[name] = umap_2d

    # HDBSCAN
    clusterer     = hdbscan.HDBSCAN(**HDBSCAN_PARAMS)
    labels        = clusterer.fit_predict(embs)
    all_labels[name] = labels

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = int((labels == -1).sum())
    print(f"  Clusters: {n_clusters} | Noise: {n_noise}")

    # Quality metrics (require ≥2 clusters and at least some non-noise points)
    non_noise_mask = labels != -1
    if n_clusters >= 2 and non_noise_mask.sum() > n_clusters:
        sil = silhouette_score(embs[non_noise_mask], labels[non_noise_mask])
        ch  = calinski_harabasz_score(embs[non_noise_mask], labels[non_noise_mask])
        db  = davies_bouldin_score(embs[non_noise_mask], labels[non_noise_mask])
    else:
        sil, ch, db = float("nan"), float("nan"), float("nan")

    print(f"  Silhouette: {sil:.4f} | CH: {ch:.2f} | DB: {db:.4f}")

    # Per-country result DataFrame
    rdf = profiles_df.copy()
    rdf["umap_x"]       = umap_2d[:, 0]
    rdf["umap_y"]       = umap_2d[:, 1]
    rdf["cluster"]      = labels
    rdf["cluster_label"] = rdf["cluster"].apply(
        lambda c: f"Cluster {c}" if c >= 0 else "Noise"
    )
    all_results[name] = rdf

    cluster_stats.append({
        "Model"            : name,
        "Dimensions"       : meta["dim"],
        "Load Time (s)"    : round(meta["load_time"], 1),
        "Encode Time (s)"  : round(meta["encode_time"], 1),
        "Num Clusters"     : n_clusters,
        "Noise Points"     : n_noise,
        "Silhouette Score" : round(sil, 4) if not np.isnan(sil) else "N/A",
        "Calinski-Harabász": round(ch, 2)  if not np.isnan(ch)  else "N/A",
        "Davies-Bouldin"   : round(db, 4)  if not np.isnan(db)  else "N/A",
    })

print("\nClustering complete for all models.")

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
summary_df = pd.DataFrame(cluster_stats)

def highlight_best(col):
    """Highlight the best-performing cell in each numeric metric column."""
    numeric = pd.to_numeric(col, errors="coerce")
    if numeric.isna().all():
        return [""] * len(col)
    if col.name == "Silhouette Score" or col.name == "Calinski-Harabász":
        best_idx = numeric.idxmax()
    elif col.name == "Davies-Bouldin":
        best_idx = numeric.idxmin()
    else:
        return [""] * len(col)
    return ["background-color: #d4f4dd; font-weight: bold"
            if i == best_idx else "" for i in col.index]

(
    summary_df.style
    .apply(highlight_best, axis=0, subset=["Silhouette Score", "Calinski-Harabász", "Davies-Bouldin"])
    .set_caption("Embedding Model Comparison — Clustering Quality Metrics")
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "14px"), ("font-weight", "bold")]}])
)

## 5. Side-by-Side UMAP Visualisations

Each panel shows the UMAP 2-D projection for one embedding model, with points coloured by
HDBSCAN cluster assignment (grey = noise).  Compare:

- **Cluster count** — do models agree on how many natural groups exist?
- **Cluster separation** — are the groups tight and well-separated or diffuse?
- **Noise fraction** — do larger/richer models assign fewer outlier points?

In [ ]:
# ── 2×3 subplot grid (5 models + 1 title panel placeholder hidden) ────────────
model_names = list(all_results.keys())
n_models    = len(model_names)
n_cols      = 3
n_rows      = (n_models + n_cols - 1) // n_cols   # ceiling division

# Build a shared colour palette so Cluster-0 maps to the same colour across panels
PALETTE = pc.qualitative.Plotly + pc.qualitative.D3 + pc.qualitative.G10

def cluster_colour_map(labels: np.ndarray) -> dict:
    unique_clusters = sorted(set(labels))
    cmap = {}
    colour_idx = 0
    for c in unique_clusters:
        if c == -1:
            cmap["Noise"] = "#cccccc"
        else:
            cmap[f"Cluster {c}"] = PALETTE[colour_idx % len(PALETTE)]
            colour_idx += 1
    return cmap

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"{nm}" for nm in model_names],
    horizontal_spacing=0.07,
    vertical_spacing=0.12,
)

for panel_idx, name in enumerate(model_names):
    row = panel_idx // n_cols + 1
    col = panel_idx  % n_cols + 1
    rdf  = all_results[name]
    cmap = cluster_colour_map(all_labels[name])

    for cluster_label, colour in cmap.items():
        subset = rdf[rdf["cluster_label"] == cluster_label]
        fig.add_trace(
            go.Scatter(
                x=subset["umap_x"],
                y=subset["umap_y"],
                mode="markers",
                marker=dict(color=colour, size=7, opacity=0.8),
                name=cluster_label,
                text=subset["country"] + "<br>Top: " + subset["top_mineral"],
                hovertemplate="%{text}<extra></extra>",
                legendgroup=cluster_label,
                showlegend=(panel_idx == 0),  # only first panel contributes legend
            ),
            row=row, col=col,
        )

fig.update_layout(
    title_text="UMAP Projections by Embedding Model (HDBSCAN clusters)",
    title_font_size=16,
    height=380 * n_rows,
    width=1100,
    template="plotly_white",
    legend=dict(title="Cluster", itemsizing="constant"),
)
fig.update_xaxes(title_text="UMAP-1", title_font_size=10)
fig.update_yaxes(title_text="UMAP-2", title_font_size=10)
fig.show()

## 6. Embedding Space Similarity Analysis

Two models might produce different cluster counts yet still **agree** on which countries are
similar to each other.  To measure this, we:

1. Compute the **pairwise cosine distance matrix** for each model
   (an N × N matrix where entry [i, j] is the distance between country i and country j).
2. Vectorise the upper triangle of each distance matrix (to avoid double-counting).
3. Compute **Spearman rank correlation** between every pair of vectorised distance matrices.

A high Spearman ρ means the two models agree on the relative ordering of country-pair
distances — i.e. they share a similar "mental map" of the mineral production landscape —
even if their absolute distance scales differ.

In [ ]:
# ── Pairwise distance matrices ────────────────────────────────────────────────
# Cosine distance = 1 − cosine similarity (on L2-normalised vectors = 1 − dot product)
dist_vectors: dict[str, np.ndarray] = {}

for name, embs in all_embeddings.items():
    sim_matrix  = cosine_similarity(embs)          # N × N similarities
    dist_matrix = 1.0 - sim_matrix                 # N × N distances
    n = dist_matrix.shape[0]
    # Upper triangle only (i < j), flattened
    tri_idx = np.triu_indices(n, k=1)
    dist_vectors[name] = dist_matrix[tri_idx]

# ── Inter-model Spearman ρ matrix ─────────────────────────────────────────────
model_list = list(dist_vectors.keys())
n_m = len(model_list)
rho_matrix = np.ones((n_m, n_m))

for i, m1 in enumerate(model_list):
    for j, m2 in enumerate(model_list):
        if i < j:
            rho, _ = spearmanr(dist_vectors[m1], dist_vectors[m2])
            rho_matrix[i, j] = rho
            rho_matrix[j, i] = rho

rho_df = pd.DataFrame(rho_matrix, index=model_list, columns=model_list)
print("Inter-model Spearman ρ (rank correlation of pairwise distances):")
print(rho_df.round(4))

In [ ]:
# ── Heatmap of inter-model agreement ─────────────────────────────────────────
fig_rho = go.Figure(
    go.Heatmap(
        z=rho_matrix,
        x=model_list,
        y=model_list,
        colorscale="RdYlGn",
        zmin=0.0, zmax=1.0,
        text=rho_df.round(3).values,
        texttemplate="%{text}",
        hovertemplate="%{y} vs %{x}: ρ = %{z:.3f}<extra></extra>",
    )
)

fig_rho.update_layout(
    title=(
        "Inter-Model Agreement: Spearman ρ of Pairwise Country Distance Matrices<br>"
        "<sup>Higher ρ means both models agree on which country pairs are similar</sup>"
    ),
    width=620,
    height=540,
    template="plotly_white",
    xaxis_title="Model",
    yaxis_title="Model",
    font=dict(size=12),
)
fig_rho.show()

In [ ]:
# ── Per-model intra-cluster vs. inter-cluster distance summary ────────────────
# For each model, compute:
#   - mean cosine distance between countries in the SAME cluster (intra)
#   - mean cosine distance between countries in DIFFERENT clusters (inter)
# A well-separated clustering shows low intra and high inter.

dist_summaries = []

for name, embs in all_embeddings.items():
    labels       = all_labels[name]
    sim_matrix   = cosine_similarity(embs)
    dist_matrix  = 1.0 - sim_matrix
    non_noise    = labels != -1

    if non_noise.sum() < 2:
        dist_summaries.append({"Model": name, "Mean Intra-cluster Dist": None,
                                "Mean Inter-cluster Dist": None, "Separation Ratio": None})
        continue

    intra_dists, inter_dists = [], []
    indices = np.where(non_noise)[0]
    for ii, i in enumerate(indices):
        for jj, j in enumerate(indices):
            if jj <= ii:
                continue
            d = dist_matrix[i, j]
            if labels[i] == labels[j]:
                intra_dists.append(d)
            else:
                inter_dists.append(d)

    intra_mean = np.mean(intra_dists) if intra_dists else float("nan")
    inter_mean = np.mean(inter_dists) if inter_dists else float("nan")
    ratio      = inter_mean / intra_mean if intra_mean > 0 else float("nan")

    dist_summaries.append({
        "Model"                  : name,
        "Mean Intra-cluster Dist": round(intra_mean, 4),
        "Mean Inter-cluster Dist": round(inter_mean, 4),
        "Separation Ratio"       : round(ratio, 3),
    })

dist_summary_df = pd.DataFrame(dist_summaries)
print("Cluster distance summary (higher separation ratio = better-separated clusters):")
dist_summary_df

## 7. Similarity Search Comparison

For a fixed **query country** ("China") we retrieve the top-10 most similar countries from
each model's embedding space and display the results side by side.

Points of interest:
- **Consensus neighbours** — countries that appear in the top-10 for *every* model are robust
  substitutes regardless of the embedding choice.
- **Model-specific neighbours** — countries that only one model ranks highly may reflect
  idiosyncratic aspects of that model's pre-training data or architecture.

In [ ]:
QUERY_COUNTRY = "China"
TOP_K         = 10


def top_k_similar(query: str, profiles: pd.DataFrame,
                  embs: np.ndarray, k: int = 10) -> list[str]:
    """Return the k most similar countries to `query` (excluding itself)."""
    mask = profiles["country"].str.lower().str.contains(query.lower())
    if mask.sum() == 0:
        raise ValueError(f"{query!r} not found in profiles.")
    idx       = profiles[mask].index[0]
    sims      = cosine_similarity(embs[idx].reshape(1, -1), embs).flatten()
    sims[idx] = -1.0   # exclude the query itself
    top_idx   = np.argsort(sims)[::-1][:k]
    return profiles.iloc[top_idx]["country"].tolist()


# Collect top-k lists for each model
sim_results: dict[str, list[str]] = {}
for name, embs in all_embeddings.items():
    try:
        sim_results[name] = top_k_similar(QUERY_COUNTRY, profiles_df, embs, TOP_K)
    except ValueError as e:
        print(f"  {name}: {e}")

# Display side-by-side table (rank → country per model)
sim_table = pd.DataFrame(
    {name: countries for name, countries in sim_results.items()},
    index=[f"Rank {i+1}" for i in range(TOP_K)],
)
print(f"Top {TOP_K} countries most similar to '{QUERY_COUNTRY}' per model:\n")
sim_table

In [ ]:
# ── Consensus analysis ────────────────────────────────────────────────────────
# Count how many models rank each country in their top-10
all_top_countries: list[str] = [c for lst in sim_results.values() for c in lst]
consensus = pd.Series(all_top_countries).value_counts().reset_index()
consensus.columns = ["Country", "Models Agreeing"]
consensus["Consensus %"] = (consensus["Models Agreeing"] / len(sim_results) * 100).round(0).astype(int)

# Which models ranked each country?
def models_that_ranked(country: str) -> str:
    return ", ".join(
        name for name, lst in sim_results.items() if country in lst
    )

consensus["Ranked by"] = consensus["Country"].apply(models_that_ranked)

print(f"\nCountries in the top-{TOP_K} for at least 2 models:")
consensus[consensus["Models Agreeing"] >= 2].to_string(index=False)

In [ ]:
# ── Heatmap: country × model membership in top-10 ────────────────────────────
candidate_countries = consensus["Country"].tolist()
heat_data = pd.DataFrame(
    {
        name: [1 if c in sim_results.get(name, []) else 0
               for c in candidate_countries]
        for name in all_embeddings
    },
    index=candidate_countries,
)

# Sort rows by total agreement (most agreed-upon at top)
heat_data = heat_data.loc[heat_data.sum(axis=1).sort_values(ascending=False).index]

fig_heat = go.Figure(
    go.Heatmap(
        z=heat_data.values,
        x=heat_data.columns.tolist(),
        y=heat_data.index.tolist(),
        colorscale=[[0, "#f5f5f5"], [1, "#2166ac"]],
        showscale=False,
        text=[["In top-10" if v else "" for v in row] for row in heat_data.values],
        texttemplate="%{text}",
        hovertemplate="%{y} — %{x}: %{text}<extra></extra>",
    )
)

fig_heat.update_layout(
    title=(
        f"Similarity Search Agreement: Countries in Top-{TOP_K} Similar to '{QUERY_COUNTRY}'<br>"
        "<sup>Blue = model ranked this country in its top-10; rows sorted by consensus</sup>"
    ),
    width=700,
    height=max(350, 30 * len(heat_data) + 120),
    template="plotly_white",
    xaxis_title="Embedding Model",
    yaxis_title="Country",
    font=dict(size=11),
    yaxis=dict(autorange="reversed"),
)
fig_heat.show()

## 8. Summary and Recommendations

### Findings

| Criterion | Likely winner | Rationale |
|---|---|---|
| **Cluster quality** (silhouette + CH + DB) | `bge-large` or `gte-large` | 1024-D encoders trained on large retrieval corpora tend to capture nuanced semantic distinctions |
| **Speed** (load + encode) | `MiniLM` | 6-layer, 384-D model; ~10× faster than large models |
| **Multilingual data** | `bge-m3` | Designed for 100+ languages; handles non-English country names gracefully |
| **Transparency / open data** | `nomic-embed` | Fully open training data and weights; Matryoshka dimensions let you trade off quality vs. speed |
| **Balanced default** | `bge-large` | Strong MTEB scores, manageable size, excellent English-language coverage |

### Decision framework

- **Production / high-stakes use** — prefer `bge-large` or `gte-large`.  Download once,
  cache locally; the extra quality is worth the ~1.3 GB cost.
- **Rapid prototyping / CPU-only machines** — `MiniLM` runs comfortably on a laptop CPU
  in seconds.  Use it for iteration, then switch to a large model for final results.
- **Datasets with non-English text** — `bge-m3` is the safe choice; multilingual pretraining
  handles accented country names and foreign-language commodity labels without preprocessing.
- **Regulatory / audit requirements** — `nomic-embed` is the only fully open-data option
  (Apache-2.0, training corpus documented); use it when reproducibility under scrutiny matters.

### Caveats

- All metrics above are computed on the **non-noise subset** only.  A model that aggressively
  places points in clusters may score higher on silhouette while actually over-clustering.
  Always inspect the UMAP visuals alongside the numbers.
- HDBSCAN hyperparameters (`min_cluster_size`, `min_samples`) interact with embedding
  geometry.  The fixed parameters used here were tuned for BGE-Large; re-tuning per model
  could change the ranking.
- The Spearman ρ inter-model agreement is a summary statistic over *all* country pairs.
  Two models can agree globally but disagree sharply for a specific commodity group of
  interest (e.g. rare earths producers).  Always run task-specific validation.

### Next steps

1. **Fine-tune** the best-performing base model on domain-specific mineral production text
   (e.g. BGS yearbook descriptions, USGS commodity summaries) to further improve cluster
   coherence.
2. **Ensemble** embeddings from two complementary models (e.g. BGE-Large + Nomic) by
   concatenation or averaging to hedge against single-model blind spots.
3. **Extend the query set** beyond China — run the similarity search for every major producer
   and compare consensus rankings to existing supply-chain vulnerability indices.